## <a name="mask_rcnn">Mask R-CNN目标检测模型</a>


### BDD100K数据处理

我们复用Mask R-CNN实例分割中的模型代码，为了减小成本，我们将BDD100K数据集中的标注信息转换成COCO数据集格式，这样，我们通过数据处理，不需要修改代码即可完成模型的训练。

#### BDD100K vs COCO

我们已经对BDD100K数据集的目录结构和标注格式有所了解，下面，我们来看一下COCO数据集的目录结构和标注格式：

COCO数据集的目录结构：

```bash
coco/
├── annotations
│   ├── instances_train2014.json
│   ├── ***.json
├── train2014
│   ├── COCO_train2014_***.jpg
└── val2014
    ├── COCO_val2014_***.jpg
```

在COCO中，图片目录的格式为"数据集类型+年份"，如"train2014"表示2014年COCO数据集的训练集。而BDD100K的格式为"images/数据集大小/数据及类型"，如"images/100k/train"表示10万图片的训练集，我们利用Linux环境的软连接机制，直接用链接来访问BDD100K：

In [ ]:
# 准备COCO数据集目录
! rm -rf data/coco
! mkdir -p data/coco/annotations

# 获取当前目录
import os
cwd = os.getcwd()

# 创建BDD100K数据集图片目录的软连接
! ln -s $cwd/data/bdd100k/images/500/train $cwd/data/coco/train2014
! ln -s $cwd/data/bdd100k/images/500/val $cwd/data/coco/val2014

# COCO数据集图片实际链接到BDD100K图片
! ls -lah ./data/coco/train2014

下面，我们使用Python代码列出图片文件

In [ ]:
from glob import glob
files = glob('./data/coco/val2014/*.jpg')[:5]
[print(f) for f in files];

from PIL import Image
Image.open(files[0])

这样，我们就可以使用代码直接访问BDD100K的图片了。

### 标注信息格式转换

我们首先来看一下COCO数据集的标注信息格式：

In [ ]:
import json

with open('./data/sample_coco_annotations.json') as f:
    coco_sample_json = json.load(f)
    
print(coco_sample_json.keys())
print(json.dumps(coco_sample_json, indent=4))

在COCO数据集中，主要字段表示的内容如下表所示：

|字段|说明|
|---|---|
|categories|数据集中所有物体的分类信息，包括id和name|
|images|数据集中所有图片信息|
|images[#].id| 图片的id|
|images[#].file_name| 图片的文件名|
|annotations|数据集中的标注信息|
|annotations[#].category_id|所标注物体的分类id|
|annotations[#].image_id|所标注物体所在图片的id|
|annotations[#].bbox|所标注物体的包围框信息|
|annotations[#].segmentation|所标注物体的实例分割信息|

接下来，我们将BDD100K转换为COCO数据集格式：

In [ ]:
from tqdm import tqdm


def bdd2coco(id_dict, labeled_images, fn):

    images = list()
    annotations = list()

    counter = 0
    for i in tqdm(labeled_images):
        counter += 1
        image = dict()
        image['file_name'] = i['name']
        image['height'] = 720
        image['width'] = 1280

        image['id'] = counter

        empty_image = True

        for l in i['labels']:
            annotation = dict()
            if l['category'] in id_dict.keys():
                empty_image = False
                annotation["iscrowd"] = 0
                annotation["image_id"] = image['id']
                x1 = l['box2d']['x1']
                y1 = l['box2d']['y1']
                x2 = l['box2d']['x2']
                y2 = l['box2d']['y2']
                annotation['bbox'] = [x1, y1, x2-x1, y2-y1]
                annotation['area'] = float((x2 - x1) * (y2 - y1))
                annotation['category_id'] = id_dict[l['category']]
                annotation['ignore'] = 0
                annotation['id'] = l['id']
                annotation['segmentation'] = [[x1, y1, x1, y2, x2, y2, x2, y1]]
                annotations.append(annotation)

        if empty_image:
            continue

        images.append(image)

    bdd_dict["images"] = images
    bdd_dict["annotations"] = annotations
    bdd_dict["type"] = "instances"
    
    json_string = json.dumps(bdd_dict)
    with open(fn, "w") as file:
        file.write(json_string)
    

bdd_dict = dict()
bdd_dict["categories"] = [
    {"supercategory": "none", "id": 1, "name": "person"},
    {"supercategory": "none", "id": 2, "name": "rider"},
    {"supercategory": "none", "id": 3, "name": "car"},
    {"supercategory": "none", "id": 4, "name": "bus"},
    {"supercategory": "none", "id": 5, "name": "truck"},
    {"supercategory": "none", "id": 6, "name": "bike"},
    {"supercategory": "none", "id": 7, "name": "motor"},
    {"supercategory": "none", "id": 8, "name": "traffic light"},
    {"supercategory": "none", "id": 9, "name": "traffic sign"},
    {"supercategory": "none", "id": 10, "name": "train"}
]


categories_dict = {i['name']: i['id'] for i in bdd_dict['categories']}

with open('./data/bdd100k/labels/bdd100k_labels_images_train.json') as f:
    bdd_train_labels = json.load(f)
    bdd2coco(categories_dict, bdd_train_labels, './data/coco/annotations/instances_train2014.json')
    
    
with open('./data/bdd100k/labels/bdd100k_labels_images_val.json') as f:
    bdd_val_labels = json.load(f)
    bdd2coco(categories_dict, bdd_val_labels, './data/coco/annotations/instances_val2014.json')    

这样，我们就将BDD100K的标注信息，转换为COCO格式，下面，我们复用Mask R-CNN的代码，直接创建COCO数据集对象。

### 使用BDD100K创建数据集对象

In [ ]:
import sys
import random
import math
import re
import time
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt

from src.mrcnn.config import Config
from src.mrcnn import utils
import src.mrcnn.model as modellib
from src.mrcnn import visualize
from src.mrcnn.model import log

%matplotlib inline 

# Directory to save logs and trained model
MODEL_DIR = "logs"

# Local path to trained weights file
BDD_MODEL_PATH = "data/mask_rcnn_bdd.h5"

class MyConfig(Config):
    # 可辨识的名称
    NAME = "myconfig"

    # GPU的数量和每个GPU处理的图片数量，可以根据实际情况进行调整，参考为Nvidia Tesla P100
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1

    # 物体的分类个数，BDD中共有10种物体+背景
    NUM_CLASSES = 1 + 10

    # 图片尺寸统一处理为1024，可以根据实际情况再进一步调小
    IMAGE_MIN_DIM = 512
    IMAGE_MAX_DIM = 512

    # 每张图片上训练的RoI个数，可以适当调小该参数
    TRAIN_ROIS_PER_IMAGE = 50

    # 每轮训练的step数量
    STEPS_PER_EPOCH = 10

    # 每轮验证的step数量
    VALIDATION_STEPS = 20
    
config = MyConfig()
# config.display()

from src.mrcnn.coco import CocoDataset
COCO_DIR = 'data/coco'

# 生成训练集
dataset_train = CocoDataset()
dataset_train.load_coco(COCO_DIR, "train", "2014") #　加载训练数据集
dataset_train.prepare()


# 生成验证集
dataset_val = CocoDataset()
dataset_val.load_coco(COCO_DIR, "val", "2014") #　加载验证数据集
dataset_val.prepare()

In [ ]:
print(dataset_train.class_names)

### 创建Mask R-CNN模型

下面，我们用training模式创建Mask R-CNN模型，并加载预训练模型：

In [ ]:
model = modellib.MaskRCNN(mode="training", config=config,
                          model_dir=MODEL_DIR)

model.load_weights(BDD_MODEL_PATH, by_name=True)

### 训练Mask R-CNN模型

接下来，我们使用准备好的BDD数据集，对模型进行训练:

In [ ]:
model.train(dataset_train, dataset_val, 
            learning_rate=config.LEARNING_RATE, 
            epochs=1, 
            layers='all')

model_savepath = './data/my_mask_rcnn_bdd.h5'
model.keras_model.save_weights(model_savepath)

## 使用Mask R-CNN检测目标

我们以inference模式创建模型对象，并加载刚刚训练好的模型文件

In [ ]:
inference_model = modellib.MaskRCNN(mode="inference", config=config,
                                    model_dir=MODEL_DIR)

inference_model.load_weights(model_savepath, by_name=True)

从验证集中随机选出一张图片，显示Ground Truth信息：

In [ ]:
print(inference_model.keras_model.input)
print(inference_model.keras_model.output)

In [ ]:
image_id = random.choice(dataset_val.image_ids)
original_image, image_meta, gt_class_id, gt_bbox, gt_mask =\
    modellib.load_image_gt(dataset_val, config, 
                           image_id, use_mini_mask=False)

log("original_image", original_image)
log("image_meta", image_meta)
log("gt_class_id", gt_class_id)
log("gt_bbox", gt_bbox)
log("gt_mask", gt_mask)

visualize.display_instances(original_image, gt_bbox, gt_mask, gt_class_id, 
                            dataset_val.class_names, show_mask=False, figsize=(8, 8))

使用模型进行预测并显示结果，同时，我们记录Mask R-CNN模型预测一张图片所用的时间：

In [ ]:
from time import time
start = time()
results = inference_model.detect([original_image], verbose=1)

duration = time() - start
print('duration of predition(in seconds): {}'.format(duration))
r = results[0]
visualize.display_instances(original_image, r['rois'], r['masks'], r['class_ids'], 
                            dataset_val.class_names, r['scores'], show_mask=False)

这样，我们就完成了一个基于BDD100K数据集的Mask R-CNN模型。在基于Region的目标检测模型中，Region的生成和预测相对比较耗时，模型精度高，但是整体速度较慢，不太适合对实时性或者速度要求比较高的场景。在下面的章节中，我们将介绍无Region(Region-free)的目标检测模型，完成高速目标的检测。